# Beaker Experiment Test Notebook

This notebook allows you to interactively test the automation framework step by step.

**Prerequisites:**
1. Start the Beaker server in a terminal: `./exec_apptainer_harmonia.sh`
2. Open the Beaker UI in a browser to watch: `http://hostname:8100/?token=...`
3. Run cells in this notebook to test automation

**Note:** You'll see messages appear in both this notebook AND the Beaker UI in real-time!

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from automation import BeakerClient, ExperimentRunner, load_config

print("Imports successful!")

## 2. Configuration

Set your server URL and token here. These should match your `.env` file.

In [ ]:
# Configuration - update these to match your setup
SERVER_URL = "http://localhost:8100"
TOKEN = "89f73481102c46c0bc13b2998f9a4fce"  # From .env JUPYTER_TOKEN

print(f"Server URL: {SERVER_URL}")
print(f"Token: {TOKEN[:10]}...")

## 3. Connect to Beaker Server

This establishes a WebSocket connection to the running Beaker server.

In [ ]:
# Create client
client = BeakerClient(
    server_url=SERVER_URL,
    token=TOKEN,
    timeout=120.0,  # 2 minute timeout for responses
)

# Connect
await client.connect()

print("Connected!")
print(f"  Kernel ID: {client.kernel_id}")
print(f"  Session ID: {client.session_id}")

## 4. Send a Single Test Message

Try sending a simple message to verify the connection works.
Watch the Beaker UI - you should see this message appear there too!

In [ ]:
# Send a simple test message
response = await client.send_message("Hello! Can you confirm you're working?")

print(f"Response type: {response.response_type}")
print(f"Duration: {response.duration_seconds:.2f}s")
print("\nAgent response:")
print("-" * 50)
print(response.content)

## 5. Test Data Harmonization

Now let's test the actual Harmonia data harmonization capabilities.

In [ ]:
# Ask to load a dataset
message = """Load the file dou.csv as a dataframe. 
Then subset the columns to just these 4: 'sample_type', 'gender', 'sample_site', 'diagnosis'"""

print(f"Sending: {message[:100]}...")
print("\n[Watch the Beaker UI to see the agent working...]\n")

response = await client.send_message(message, timeout=120.0)

print(f"Response type: {response.response_type}")
print(f"Duration: {response.duration_seconds:.2f}s")
print("\nAgent response:")
print("-" * 50)
print(response.content[:500] if len(response.content) > 500 else response.content)

In [ ]:
# Ask for schema matching
message = """Please match this dataframe to the GDC schema using the embeddings method.
Show me the top 3 matching columns for each source column."""

print(f"Sending: {message}")
print("\n[This may take a while - watch the Beaker UI...]\n")

response = await client.send_message(message, timeout=300.0)  # 5 minute timeout

print(f"Response type: {response.response_type}")
print(f"Duration: {response.duration_seconds:.2f}s")
print("\nAgent response:")
print("-" * 50)
print(response.content)

## 6. Run a Complete Experiment from Config

Now let's test running a complete experiment from a YAML config file.

In [ ]:
# Load experiment config
config_path = project_root / "experiments" / "configs" / "dou_harmonization.yaml"

if config_path.exists():
    config = load_config(config_path)
    print(f"Loaded experiment: {config.name}")
    print(f"  Description: {config.description}")
    print(f"  LLM: {config.llm.provider}/{config.llm.model}")
    print(f"  Messages: {len(config.messages)}")

    # Show first few messages
    print("\nFirst messages:")
    for i, msg in enumerate(config.messages[:3], 1):
        preview = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
        print(f"  {i}. {preview}")
else:
    print(f"Config not found: {config_path}")
    print("Create the config file first, or use a different path.")

In [ ]:
# Run experiment in interactive mode (pauses between turns)
if config_path.exists():
    runner = ExperimentRunner(
        client=client,
        config=config,
        output_dir=project_root / "results",
    )

    print("Running experiment in interactive mode...")
    print("(Press Enter after each turn to continue)\n")

    output_dir = await runner.run(interactive=True)

    print("\nExperiment complete!")
    print(f"Output directory: {output_dir}")

## 7. View Results

In [ ]:
# List result files
results_dir = project_root / "results"

if results_dir.exists():
    print("Result directories:")
    for d in sorted(results_dir.iterdir()):
        if d.is_dir():
            print(f"  {d.name}/")
            for f in d.iterdir():
                print(f"    - {f.name}")
else:
    print("No results directory yet. Run an experiment first!")

In [ ]:
# View conversation.md from most recent result
if results_dir.exists():
    latest = sorted(results_dir.iterdir())[-1] if list(results_dir.iterdir()) else None
    if latest:
        conv_file = latest / "conversation.md"
        if conv_file.exists():
            print(f"=== {conv_file} ===")
            print(conv_file.read_text())

## 8. Cleanup

In [ ]:
# Disconnect from server
await client.disconnect()
print("Disconnected from Beaker server.")

## Troubleshooting

**Connection refused:**
- Make sure Beaker server is running: `./exec_apptainer_harmonia.sh`
- Check the server URL and port

**Authentication error:**
- Verify the token matches `JUPYTER_TOKEN` in your `.env` file

**Timeout:**
- Increase the timeout parameter
- Check if the agent is still processing in the Beaker UI

**Messages not appearing in Beaker UI:**
- Refresh the Beaker UI page
- Check that you're connected to the same kernel